In [ ]:
from __future__ import annotations

import math
from pathlib import Path

from matplotlib import pyplot as plt

import vascx.fundus.feature_sets  # noqa: F401 — registers FeatureSets
from vascx.fundus.features.base import LayerFeature, RetinaFeature, VesselsLayerFeature
from vascx.fundus.loader import RetinaLoader
from vascx.shared.features import FeatureSet

## Feature set visualisation

This notebook plots all the biomarker visualisations for a single image and feature set. Many will likely be very similar or repeated because most feature sets have variants of the same feature.

Note: it assummes that the previous notebooks have been run.

In [ ]:
ds_path = Path("../samples/fundus")
FEATURE_SET_NAME = "full_v3"
IMID = "HRF_07_dr"
NCOLS = 3

In [ ]:
def plot_target_for_feature(feature, retina):
    """Layer features use arteries only; vessels features use the vessels layer."""
    if isinstance(feature, RetinaFeature):
        return retina, "retina"
    if isinstance(feature, LayerFeature):
        return retina.arteries, "arteries"
    if isinstance(feature, VesselsLayerFeature):
        return retina.vessels, "vessels"
    raise TypeError(f"Unsupported feature type: {type(feature)}")


feature_set = FeatureSet.get_by_name(FEATURE_SET_NAME)
if feature_set is None:
    raise ValueError(f"Unknown feature set: {FEATURE_SET_NAME!r}")

loader = RetinaLoader.from_folder(ds_path)
retina = loader.by_id(IMID)

n_features = len(feature_set)
nrows = int(math.ceil(n_features / NCOLS))


fig, axs = plt.subplots(
    nrows,
    NCOLS,
    figsize=(NCOLS * 3.0, nrows * 3.0),
    dpi=120,
    squeeze=False,
)
for i, ax in enumerate(axs.flat):
    if i >= n_features:
        ax.axis("off")
        continue
    feat = feature_set.features[i]
    target, layer_name = plot_target_for_feature(feat, retina)
    feat.plot(ax, target)
    ax.set_title(feat.display_name(layer_name=layer_name), fontsize=7)
    ax.set_axis_off()

fig.suptitle(f"{FEATURE_SET_NAME} — {n_features} features", fontsize=12, y=1.0)
fig.tight_layout(
    rect=(0.0, 0.0, 1.0, 0.96),
    pad=0.0,
    h_pad=0.05,
    w_pad=0.05,
)